# Statoil Iceberg — Fixed Best Model Submission

**Goal:** Generate `/kaggle/working/submission.csv` using the **fixed best configuration**, previously selected by the SMAC search. **No SMAC tuning or hyperparameter search runs in this Notebook.**

Model: **EfficientNet-B0 + incidence angle**, Rotation augmentation, Adam (`lr=0.0013541313614`, `weight_decay=0`), Dropout `0.0933272827915`, Label Smoothing `0.0033914502723`, cosine scheduler, early stopping.

**Previous validation reference:** 5-fold mean CV Loss = **0.187174** (not a Kaggle test score). This submission fits **five copies of that same fixed configuration**, and averages their test-set probabilities. The numerical CV loss of a new run may vary slightly.

Add the Kaggle competition dataset, enable GPU and ensure pretrained EfficientNet weights are accessible. Run each code cell top-to-bottom.

In [1]:
from pathlib import Path
import subprocess
import random
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models

from sklearn.model_selection import StratifiedKFold

from torch.utils.data import Dataset, DataLoader

from torchvision.transforms import (
    Compose,
    Normalize,
)


SEED = 42
BATCH_SIZE = 32
NUM_FOLDS = 5

MAX_EPOCHS = 30
INITIAL_LR = 0.0013541313614
MIN_LR = 0.00001
PATIENCE = 8

NUM_WORKERS = 2

# Set False if pretrained EfficientNet weights are unavailable.
USE_PRETRAINED = True

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


set_seed(SEED)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

# Fixed final parameters selected by the prior search (no SMAC runs here).
FINAL_CONFIG = {
    "augmentation": "rotation",
    "model_name": "efficientnet",
    "optimizer_name": "adam",
    "use_scheduler": True,
    "initial_lr": 0.0013541313614,
    "weight_decay": 0.0,
    "dropout_p": 0.0933272827915,
    "label_smoothing": 0.0033914502723,
    "use_angle": True,
    "use_early_stopping": True,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "min_lr": MIN_LR,
    "patience": PATIENCE,
    "seed": SEED,
    "n_folds": NUM_FOLDS,
    "pretrained": USE_PRETRAINED,
}

# Completed fold checkpoints can be reused if the Kaggle working files persist.
SAVE_FOLD_CHECKPOINTS = True
REUSE_VALID_CHECKPOINTS = True

Device: cuda
GPU: Tesla T4


## Load and preprocess the competition data

In [2]:
import os
import json
import gc
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/iceberg-submission")
WORK_DIR.mkdir(parents=True, exist_ok=True)


def locate_dataset_file(file_name):
    # Favor extracted files in the working directory, then attached Kaggle data.
    for base in (WORK_DIR, INPUT_ROOT):
        found = sorted(base.rglob(file_name))
        if found:
            return found[0]

    archive_name = file_name + ".7z"
    archives = sorted(INPUT_ROOT.rglob(archive_name))
    if not archives:
        raise FileNotFoundError(
            f"Could not find {file_name} or {archive_name}. "
            "Add the competition dataset via Kaggle's Add Input."
        )

    subprocess.run(
        ["7z", "x", str(archives[0]), f"-o{WORK_DIR}", "-y"],
        check=True,
    )
    found = sorted(WORK_DIR.rglob(file_name))
    if not found:
        raise FileNotFoundError(f"Extraction produced no {file_name}")
    return found[0]


train_path = locate_dataset_file("train.json")
test_path = locate_dataset_file("test.json")
train_df = pd.read_json(train_path)
test_df = pd.read_json(test_path)

required_train = {"id", "band_1", "band_2", "inc_angle", "is_iceberg"}
required_test = {"id", "band_1", "band_2", "inc_angle"}
assert required_train.issubset(train_df.columns), "Missing train.json columns"
assert required_test.issubset(test_df.columns), "Missing test.json columns"
assert train_df["id"].is_unique and test_df["id"].is_unique

print("Train examples:", len(train_df))
print("Test examples:", len(test_df))
print("Train file:", train_path)
print("Test file:", test_path)

def convert_images_and_angles(df):
    angles = pd.to_numeric(df["inc_angle"], errors="coerce").to_numpy(
        dtype=np.float32
    ).reshape(-1, 1)
    angle_missing = np.isnan(angles).astype(np.float32)

    band_1 = np.stack([
        np.asarray(band, dtype=np.float32).reshape(75, 75)
        for band in df["band_1"]
    ])
    band_2 = np.stack([
        np.asarray(band, dtype=np.float32).reshape(75, 75)
        for band in df["band_2"]
    ])
    band_3 = (band_1 + band_2) / 2.0

    images = np.stack([band_1, band_2, band_3], axis=1).astype(np.float32)
    ids = df["id"].astype(str).to_numpy()
    return ids, images, angles, angle_missing


train_ids, X_train_img, X_train_ang, X_train_ang_missing = (
    convert_images_and_angles(train_df)
)
test_ids, X_test_img, X_test_ang, X_test_ang_missing = (
    convert_images_and_angles(test_df)
)
Y_train = train_df["is_iceberg"].to_numpy(dtype=np.float32)

# Drop heavy raw JSON arrays once converted to tensors.
train_df = train_df[["id", "is_iceberg"]].copy()
del test_df
gc.collect()

assert X_train_img.shape[1:] == (3, 75, 75)
assert X_test_img.shape[1:] == (3, 75, 75)
assert len(train_ids) == len(Y_train)
print("Training images:", X_train_img.shape)
print("Test images:", X_test_img.shape)

splitter = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=SEED,
)

fold_numbers = np.full(
    len(train_df),
    fill_value=-1,
    dtype=np.int64,
)

for fold, (_, val_indices) in enumerate(
    splitter.split(
        train_df,
        train_df["is_iceberg"]
    )
):

    fold_numbers[
        val_indices
    ] = fold

train_df[
    "fold"
] = fold_numbers


print("Samples in each fold:")

print(
    train_df[
        "fold"
    ].value_counts().sort_index()
)

print(
    "\nLabel distribution:"
)

print(
    pd.crosstab(
        train_df["fold"],
        train_df["is_iceberg"],
    )
)


7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:4 OPEN_MAX:1048576

Scanning the drive for archives:
1 file, 44932785 bytes (43 MiB)

Extracting archive: /kaggle/input/competitions/statoil-iceberg-classifier-challenge/train.json.7z
--
Path = /kaggle/input/competitions/statoil-iceberg-classifier-challenge/train.json.7z
Type = 7z
Physical Size = 44932785
Headers Size = 154
Method = LZMA2:24
Solid = -
Blocks = 1

Everything is Ok

Size:       196313674
Compressed: 44932785

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:4 OPEN_MAX:1048576

Scanning the drive for archives:
1 file, 257127394 bytes (246 MiB)

Extracting archive: /kaggle/input/competitions/statoil-iceberg-classifier-challenge/test.json.7z
--
Path = /kaggle/input/competitions/statoil-iceberg-classifier-challenge/test.json.7z
Type = 7z
Physical Size = 257127394
Headers Size = 154
Method = LZMA2:24
Solid = -
Blocks =

## Dataset, transformations, and model

In [3]:
class IcebergDataset(Dataset):

    """Dataset for SAR images, incidence angles, labels, and IDs."""

    def __init__(
        self,
        images,
        angles,
        angle_missing,
        ids,
        indices,
        angle_mean,
        angle_std,
        labels=None,
        transform=None,
    ):

        self.images = images
        self.angles = angles
        self.angle_missing = angle_missing
        self.ids = ids

        self.indices = np.asarray(
            indices
        )

        self.angle_mean = float(
            angle_mean
        )

        self.angle_std = float(
            angle_std
        )

        if self.angle_std <= 0:

            raise ValueError(
                "angle_std must be greater than 0"
            )

        self.labels = labels
        self.transform = transform


    def __len__(self):

        return len(
            self.indices
        )


    def __getitem__(self, index):

        original_index = (
            self.indices[index]
        )

        image = torch.from_numpy(
            self.images[
                original_index
            ].copy()
        ).float()

        if self.transform is not None:

            image = self.transform(
                image
            )

        angle = float(
            self.angles[
                original_index,
                0
            ]
        )

        missing = float(
            self.angle_missing[
                original_index,
                0
            ]
        )

        # Fill missing angles with the training-fold mean.
        if np.isnan(angle):

            angle = self.angle_mean

        # Standardize using training-fold statistics.
        angle = (
            angle - self.angle_mean
        ) / self.angle_std

        sample = {
            "image": image,

            "angle": torch.tensor(
                [angle],
                dtype=torch.float32,
            ),

            "angle_missing": torch.tensor(
                [missing],
                dtype=torch.float32,
            ),

            "id": str(
                self.ids[
                    original_index
                ]
            ),
        }

        if self.labels is not None:

            sample[
                "target"
            ] = torch.tensor(
                self.labels[
                    original_index
                ],
                dtype=torch.float32,
            )

        return sample

class RandomRotate90:
    def __call__(self, image):
        k = int(torch.randint(0, 4, (1,)).item())
        return torch.rot90(image, k=k, dims=(-2, -1))


def build_train_transform(image_mean, image_std, augmentation="rotation"):
    if augmentation != "rotation":
        raise ValueError("This submission uses rotation augmentation only")
    return Compose([
        RandomRotate90(),
        Normalize(mean=image_mean.tolist(), std=image_std.tolist()),
    ])


def build_val_transform(image_mean, image_std):
    return Normalize(mean=image_mean.tolist(), std=image_std.tolist())


In [4]:
class IcebergModel(nn.Module):
    def __init__(self, use_pretrained=True, dropout_p=0.4, use_angle=True):
        super().__init__()
        if not 0.0 <= dropout_p < 1.0:
            raise ValueError("dropout_p must be between 0 and 1")
        self.use_angle = use_angle
        weights = (
            models.EfficientNet_B0_Weights.DEFAULT
            if use_pretrained else None
        )
        self.image_tower = models.efficientnet_b0(weights=weights)
        self.image_tower.classifier = nn.Identity()
        self.angle_tower = nn.Sequential(nn.Linear(1, 16), nn.ReLU())
        self.fusion_head = nn.Sequential(
            nn.Linear(1296, 256),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, image, angle):
        image = F.interpolate(
            image, size=(224, 224), mode="bilinear", align_corners=False
        )
        image_features = self.image_tower(image)
        if self.use_angle:
            angle_features = self.angle_tower(angle)
        else:
            # Keep fusion dimensions unchanged, but remove angle information.
            angle_features = torch.zeros(
                image_features.size(0), 16,
                device=image_features.device,
                dtype=image_features.dtype,
            )
        features = torch.cat([image_features, angle_features], dim=1)
        return self.fusion_head(features)


def create_model(model_name="efficientnet", dropout_p=0.0933272827915, use_angle=True):
    if model_name != "efficientnet":
        raise ValueError("This submission uses EfficientNet-B0 only")
    return IcebergModel(
        use_pretrained=USE_PRETRAINED,
        dropout_p=dropout_p,
        use_angle=use_angle,
    )


def create_fold_loaders(
    fold,
    augmentation="baseline",
    batch_size=BATCH_SIZE,
    seed=SEED,
):

    fold_array = train_df[
        "fold"
    ].to_numpy()

    val_indices = np.where(
        fold_array == fold
    )[0]

    train_indices = np.where(
        fold_array != fold
    )[0]

    # Compute angle statistics from training data only.
    angle_mean = np.nanmean(
        X_train_ang[
            train_indices,
            0
        ]
    )

    angle_std = np.nanstd(
        X_train_ang[
            train_indices,
            0
        ]
    )

    # Compute image statistics from training data only.
    train_images = X_train_img[
        train_indices
    ]

    image_mean = train_images.mean(
        axis=(0, 2, 3),
        dtype=np.float64
    ).astype(np.float32)

    image_std = train_images.std(
        axis=(0, 2, 3),
        dtype=np.float64
    ).astype(np.float32)

    train_transform = (
        build_train_transform(
            image_mean,
            image_std,
            augmentation,
        )
    )

    val_transform = (
        build_val_transform(
            image_mean,
            image_std,
        )
    )

    train_dataset = IcebergDataset(
        images=X_train_img,
        angles=X_train_ang,
        angle_missing=X_train_ang_missing,
        ids=train_ids,
        indices=train_indices,
        angle_mean=angle_mean,
        angle_std=angle_std,
        labels=Y_train,
        transform=train_transform,
    )

    val_dataset = IcebergDataset(
        images=X_train_img,
        angles=X_train_ang,
        angle_missing=X_train_ang_missing,
        ids=train_ids,
        indices=val_indices,
        angle_mean=angle_mean,
        angle_std=angle_std,
        labels=Y_train,
        transform=val_transform,
    )

    generator = torch.Generator()

    generator.manual_seed(
        seed
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
        generator=generator,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
    )

    return (
        train_loader,
        val_loader
    )

## Train with fixed parameters, one fold at a time

In [5]:
def train_one_epoch(
    model, train_loader, criterion, optimizer, device, label_smoothing=0.0
):
    if not 0.0 <= label_smoothing < 1.0:
        raise ValueError("label_smoothing must be in [0, 1)")
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        images = batch["image"].to(device, non_blocking=True)
        angles = batch["angle"].to(device, non_blocking=True)
        targets = batch["target"].unsqueeze(1).to(device, non_blocking=True)

        # Binary label smoothing: 0 -> eps/2, 1 -> 1 - eps/2.
        training_targets = targets * (1.0 - label_smoothing) + 0.5 * label_smoothing

        optimizer.zero_grad()
        predictions = model(images, angles)
        loss = criterion(predictions, training_targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    return running_loss / len(train_loader.dataset)


def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device, non_blocking=True)
            angles = batch["angle"].to(device, non_blocking=True)
            targets = batch["target"].unsqueeze(1).to(device, non_blocking=True)
            predictions = model(images, angles)
            # Validation always uses original 0/1 labels.
            loss = criterion(predictions, targets)
            running_loss += loss.item() * images.size(0)
    return running_loss / len(val_loader.dataset)


def train_one_fold(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, max_epochs, patience,
    use_scheduler=True, use_early_stopping=True, label_smoothing=0.0,
):
    best_val_loss = float("inf")
    best_train_loss = None
    best_epoch = 0
    best_model_state = None
    epochs_without_improvement = 0
    final_train_loss = None
    final_val_loss = None
    final_epoch = 0

    for epoch in range(max_epochs):
        current_lr = optimizer.param_groups[0]["lr"]
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device,
            label_smoothing=label_smoothing,
        )
        val_loss = validate(model, val_loader, criterion, device)
        print(
            f"Epoch {epoch + 1:02d} | Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | LR: {current_lr:.6f}",
            flush=True,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_train_loss = train_loss
            best_epoch = epoch + 1
            if use_early_stopping:
                best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        final_train_loss = train_loss
        final_val_loss = val_loss
        final_epoch = epoch + 1

        if use_early_stopping and epochs_without_improvement >= patience:
            print("Early stopping triggered.", flush=True)
            break
        if use_scheduler and scheduler is not None:
            scheduler.step()

    if use_early_stopping:
        selected_train_loss = best_train_loss
        selected_val_loss = best_val_loss
        selected_epoch = best_epoch
        selected_state = best_model_state
    else:
        selected_train_loss = final_train_loss
        selected_val_loss = final_val_loss
        selected_epoch = final_epoch
        selected_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(selected_state)
    return selected_train_loss, selected_val_loss, selected_epoch, selected_state

In [6]:
def get_fold_statistics(fold):
    fold_array = train_df["fold"].to_numpy()
    train_indices = np.where(fold_array != fold)[0]

    angle_mean = float(np.nanmean(X_train_ang[train_indices, 0]))
    angle_std = float(np.nanstd(X_train_ang[train_indices, 0]))
    train_images = X_train_img[train_indices]
    image_mean = train_images.mean(
        axis=(0, 2, 3), dtype=np.float64
    ).astype(np.float32)
    image_std = train_images.std(
        axis=(0, 2, 3), dtype=np.float64
    ).astype(np.float32)

    if not np.isfinite(angle_std) or angle_std <= 0:
        raise ValueError(f"Invalid angle standard deviation for fold {fold}")
    if not np.isfinite(image_std).all() or np.any(image_std <= 0):
        raise ValueError(f"Invalid image standard deviation for fold {fold}")
    return angle_mean, angle_std, image_mean, image_std


def create_test_loader(stats):
    angle_mean, angle_std, image_mean, image_std = stats
    test_dataset = IcebergDataset(
        images=X_test_img,
        angles=X_test_ang,
        angle_missing=X_test_ang_missing,
        ids=test_ids,
        indices=np.arange(len(test_ids)),
        angle_mean=angle_mean,
        angle_std=angle_std,
        labels=None,
        transform=build_val_transform(image_mean, image_std),
    )
    return DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
    )


@torch.inference_mode()
def predict_probabilities(model, data_loader):
    model.eval()
    outputs = []
    for batch in data_loader:
        image = batch["image"].to(device, non_blocking=True)
        angle = batch["angle"].to(device, non_blocking=True)
        prob = model(image, angle).flatten().cpu().numpy()
        outputs.append(prob)
    return np.concatenate(outputs).astype(np.float64)


def load_verified_fold_checkpoint(path, fold):
    if not path.is_file() or not REUSE_VALID_CHECKPOINTS:
        return None
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=True)
        if (checkpoint.get("config") != FINAL_CONFIG
                or int(checkpoint.get("fold", -1)) != fold):
            print(f"[SKIP CHECKPOINT] Fold {fold}: configuration differs")
            return None
        required = {"state_dict", "val_loss", "selected_epoch"}
        if not required.issubset(checkpoint):
            return None
        return checkpoint
    except Exception as e:
        print(f"[SKIP CHECKPOINT] Fold {fold}: {e}")
        return None


fold_predictions = []
fold_details = []

for fold in range(NUM_FOLDS):
    print(f"\n========== SUBMISSION FOLD {fold} ==========" , flush=True)
    set_seed(SEED + fold)
    checkpoint_path = WORK_DIR / f"fixed_best_fold_{fold}.pt"
    checkpoint = load_verified_fold_checkpoint(checkpoint_path, fold)
    stats = get_fold_statistics(fold)

    if checkpoint is None:
        train_loader, val_loader = create_fold_loaders(
            fold=fold,
            augmentation=FINAL_CONFIG["augmentation"],
            batch_size=BATCH_SIZE,
            seed=SEED + fold,
        )
        try:
            model = create_model(
                FINAL_CONFIG["model_name"],
                dropout_p=FINAL_CONFIG["dropout_p"],
                use_angle=FINAL_CONFIG["use_angle"],
            ).to(device)
        except Exception as exc:
            raise RuntimeError(
                "EfficientNet-B0 pretrained weights could not be loaded. "
                "Make the pretrained checkpoint available before rerunning; "
                "do NOT switch to random initialization if reproducing SMAC."
            ) from exc

        criterion = nn.BCELoss()
        optimizer = optim.Adam(
            model.parameters(),
            lr=FINAL_CONFIG["initial_lr"],
            weight_decay=FINAL_CONFIG["weight_decay"],
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=MAX_EPOCHS,
            eta_min=MIN_LR,
        )
        train_loss, val_loss, selected_epoch, state = train_one_fold(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            max_epochs=MAX_EPOCHS,
            patience=PATIENCE,
            use_scheduler=True,
            use_early_stopping=True,
            label_smoothing=FINAL_CONFIG["label_smoothing"],
        )

        checkpoint = {
            "config": FINAL_CONFIG.copy(),
            "fold": int(fold),
            "state_dict": {k: v.detach().cpu() for k, v in state.items()},
            "selected_epoch": int(selected_epoch),
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
        }
        if SAVE_FOLD_CHECKPOINTS:
            temporary_path = checkpoint_path.with_suffix(".pt.tmp")
            torch.save(checkpoint, temporary_path)
            os.replace(temporary_path, checkpoint_path)
            print(f"[SAVED] Fold {fold}: {checkpoint_path}")

        # Reuse this model for test inference, but do not retain all fold models.
        del train_loader, val_loader, optimizer, scheduler, criterion, state
    else:
        print(f"[RESTORED] Fold {fold}: {checkpoint_path}")
        # Checkpoint contains full trained weights; no ImageNet download is needed.
        model = IcebergModel(
            use_pretrained=False,
            dropout_p=FINAL_CONFIG["dropout_p"],
            use_angle=FINAL_CONFIG["use_angle"],
        ).to(device)
        model.load_state_dict(checkpoint["state_dict"])

    test_loader = create_test_loader(stats)
    preds = predict_probabilities(model, test_loader)
    if len(preds) != len(test_ids) or not np.isfinite(preds).all():
        raise ValueError(f"Invalid test predictions for fold {fold}")
    fold_predictions.append(preds)

    fold_details.append({
        "fold": fold,
        "selected_epoch": int(checkpoint["selected_epoch"]),
        "train_loss": float(checkpoint.get("train_loss", np.nan)),
        "val_loss": float(checkpoint["val_loss"]),
        "checkpoint": str(checkpoint_path) if SAVE_FOLD_CHECKPOINTS else "not saved",
    })
    print(
        f"Fold {fold}: selected epoch={checkpoint['selected_epoch']}; "
        f"val_loss={checkpoint['val_loss']:.6f}",
        flush=True,
    )

    del model, test_loader, checkpoint, preds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

fold_details_df = pd.DataFrame(fold_details)
fold_details_df.to_csv(WORK_DIR / "submission_fold_results.csv", index=False)

print("\n5-fold summary:")
print(fold_details_df.to_string(index=False))
print("Mean CV Loss:", round(fold_details_df["val_loss"].mean(), 6))
print("Prior best 5-fold CV Loss: 0.187174 (reference only)")


========== SUBMISSION FOLD 0 ==========
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 124MB/s]


Epoch 01 | Train Loss: 0.4877 | Val Loss: 0.3483 | LR: 0.001354
Epoch 02 | Train Loss: 0.3246 | Val Loss: 0.2823 | LR: 0.001350
Epoch 03 | Train Loss: 0.2890 | Val Loss: 0.3092 | LR: 0.001339
Epoch 04 | Train Loss: 0.2818 | Val Loss: 0.3085 | LR: 0.001321
Epoch 05 | Train Loss: 0.2543 | Val Loss: 0.5257 | LR: 0.001296
Epoch 06 | Train Loss: 0.2751 | Val Loss: 0.2780 | LR: 0.001264
Epoch 07 | Train Loss: 0.2576 | Val Loss: 0.3007 | LR: 0.001226
Epoch 08 | Train Loss: 0.2914 | Val Loss: 0.2676 | LR: 0.001182
Epoch 09 | Train Loss: 0.2292 | Val Loss: 0.3660 | LR: 0.001132
Epoch 10 | Train Loss: 0.2313 | Val Loss: 0.2004 | LR: 0.001077
Epoch 11 | Train Loss: 0.2365 | Val Loss: 0.2256 | LR: 0.001018
Epoch 12 | Train Loss: 0.2088 | Val Loss: 0.2554 | LR: 0.000955
Epoch 13 | Train Loss: 0.1889 | Val Loss: 0.2287 | LR: 0.000890
Epoch 14 | Train Loss: 0.1910 | Val Loss: 0.2386 | LR: 0.000822
Epoch 15 | Train Loss: 0.1522 | Val Loss: 0.2434 | LR: 0.000752
Epoch 16 | Train Loss: 0.1371 | Val Loss

## Write Kaggle predictions

In [7]:
prediction_matrix = np.stack(fold_predictions, axis=0)
mean_test_prob = prediction_matrix.mean(axis=0)
mean_test_prob = np.clip(mean_test_prob, 1e-7, 1.0 - 1e-7)

submission = pd.DataFrame({
    "id": test_ids.astype(str),
    "is_iceberg": mean_test_prob,
})

# Use the competition's official sample IDs/order when available.
try:
    sample_path = locate_dataset_file("sample_submission.csv")
except FileNotFoundError:
    sample_path = None

if sample_path is not None:
    sample = pd.read_csv(sample_path, dtype={"id": str})
    assert list(sample.columns) == ["id", "is_iceberg"], sample.columns.tolist()
    assert sample["id"].is_unique, "Duplicate IDs in sample submission"
    assert set(sample["id"]) == set(submission["id"]), "Test and sample IDs differ"
    submission = sample[["id"]].merge(
        submission, on="id", how="left", validate="one_to_one"
    )

assert submission.columns.tolist() == ["id", "is_iceberg"]
assert submission["id"].is_unique
assert submission["is_iceberg"].notna().all()
assert submission["is_iceberg"].between(0, 1).all()
assert len(submission) == len(test_ids)

submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False, float_format="%.10f")

config_path = WORK_DIR / "final_parameters.json"
config_path.write_text(json.dumps(FINAL_CONFIG, indent=2), encoding="utf-8")

print(f"\nSUCCESS: {submission_path}")
print("Rows:", len(submission))
print("Prediction range:", float(mean_test_prob.min()), float(mean_test_prob.max()))
print(submission.head().to_string(index=False))
print("\nUpload /kaggle/working/submission.csv via Kaggle > Submit Predictions.")


7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:4 OPEN_MAX:1048576

Scanning the drive for archives:
1 file, 38566 bytes (38 KiB)

Extracting archive: /kaggle/input/competitions/statoil-iceberg-classifier-challenge/sample_submission.csv.7z
--
Path = /kaggle/input/competitions/statoil-iceberg-classifier-challenge/sample_submission.csv.7z
Type = 7z
Physical Size = 38566
Headers Size = 178
Method = LZMA2:17
Solid = -
Blocks = 1

Everything is Ok

Size:       117951
Compressed: 38566

SUCCESS: /kaggle/working/submission.csv
Rows: 8424
Prediction range: 0.0004888074376140139 0.9999983787536622
      id  is_iceberg
5941774d    0.059208
4023181e    0.252659
b20200e4    0.056536
e7f018bb    0.998041
4371c8c3    0.997630

Upload /kaggle/working/submission.csv via Kaggle > Submit Predictions.


**Submission:** Download `/kaggle/working/submission.csv` and upload that CSV via the Kaggle competition’s submission interface (if submissions are open). Do not upload the Notebook as the prediction file.

The notebook saves per-fold checkpoints under `/kaggle/working/iceberg-submission/` to make it possible to reuse completed folds if their files persist. A previously saved CSV of validation results alone is not enough to generate test predictions.